In [1]:
import os, json, time, random
from collections import defaultdict
import numpy as np
import cv2
import torch
import torch.nn as nn
from PIL import Image
from transformers import VideoMAEModel, VideoMAEImageProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_ID = "MCG-NJU/videomae-base"
NUM_FRAMES = 16
FEATURE_CACHE_DIR = "/kaggle/working/features_facecrop"
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)

def find_dataset_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "original" in dirs and "Deepfakes" in dirs:
            return root
    raise FileNotFoundError("dataset not found")

DATASET_ROOT = find_dataset_root()

manifest_path = "/kaggle/working/video_split_manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path) as fh:
        manifest = json.load(fh)
else:
    print("Manifest missing (session reset) — rebuilding identical split (seed=42)...")
    FAKE_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
    REAL_DIR = "original"
    SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
    SEED = 42

    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    def real_id(f): return os.path.splitext(f)[0]
    def parse_pair(f):
        s, t = os.path.splitext(f)[0].split("_"); return s, t

    real_ids = sorted(real_id(f) for f in os.listdir(os.path.join(DATASET_ROOT, REAL_DIR)) if f.endswith(".mp4"))
    for rid in real_ids: find(rid)

    fake_files = defaultdict(list)
    for method in FAKE_METHODS:
        for f in os.listdir(os.path.join(DATASET_ROOT, method)):
            if f.endswith(".mp4"):
                fake_files[method].append(f)
                s, t = parse_pair(f); union(s, t)

    components = defaultdict(list)
    for rid in real_ids: components[find(rid)].append(rid)
    comp_list = list(components.items())
    random.Random(SEED).shuffle(comp_list)

    n_real = len(real_ids)
    targets = {k: v * n_real for k, v in SPLIT_RATIOS.items()}
    assigned = {"train": [], "val": [], "test": []}
    counts = {"train": 0, "val": 0, "test": 0}
    for comp_root, ids in comp_list:
        deficits = {k: targets[k] - counts[k] for k in targets}
        best = max(deficits, key=lambda k: deficits[k])
        assigned[best].extend(ids); counts[best] += len(ids)
    id_to_split = {rid: s for s, ids in assigned.items() for rid in ids}

    manifest = {"train": [], "val": [], "test": []}
    for rid in real_ids:
        manifest[id_to_split[rid]].append({"path": os.path.join(REAL_DIR, f"{rid}.mp4"), "label": 1, "method": "original"})
    for method in FAKE_METHODS:
        for f in fake_files[method]:
            s, t = parse_pair(f)
            s_split, t_split = id_to_split.get(s), id_to_split.get(t)
            if s_split == t_split:
                manifest[s_split].append({"path": os.path.join(method, f), "label": 0, "method": method})

    with open(manifest_path, "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Rebuilt manifest saved.")

print({k: len(v) for k, v in manifest.items()})

_face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def crop_to_face_np(frame_rgb):
    try:
        gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
        faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        if len(faces) == 0:
            return frame_rgb
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        mx, my = int(w * 0.2), int(h * 0.2)
        ih, iw = frame_rgb.shape[:2]
        x0, y0 = max(0, x - mx), max(0, y - my)
        x1, y1 = min(iw, x + w + mx), min(ih, y + h + my)
        return frame_rgb[y0:y1, x0:x1]
    except Exception:
        return frame_rgb

def extract_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    target_idxs = set(np.linspace(0, max(total - 1, 0), num_frames, dtype=int).tolist())
    frames, idx = [], 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if idx in target_idxs:
            ok2, frame = cap.retrieve()
            if ok2:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(crop_to_face_np(rgb))
        idx += 1
        if len(frames) >= num_frames:
            break
    cap.release()
    if len(frames) < num_frames:
        if not frames:
            return None
        while len(frames) < num_frames:
            frames.append(frames[-1])
    return frames[:num_frames]

processor = VideoMAEImageProcessor.from_pretrained(MODEL_ID)
backbone = VideoMAEModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False

@torch.no_grad()
def embed_video(video_path):
    frames = extract_frames(video_path)
    if frames is None:
        return None
    inputs = processor(images=frames, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)
    if pixel_values.dim() == 4:
        pixel_values = pixel_values.unsqueeze(0)
    out = backbone(pixel_values=pixel_values)
    return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

def extract_split_features(split_name, entries):
    cache_path = os.path.join(FEATURE_CACHE_DIR, f"{split_name}.npz")
    if os.path.exists(cache_path):
        print(f"[{split_name}] cache found, loading")
        d = np.load(cache_path)
        return d["X"], d["y"]
    X, y, failures = [], [], []
    t0 = time.time()
    for i, entry in enumerate(entries):
        vec = embed_video(os.path.join(DATASET_ROOT, entry["path"]))
        if vec is None:
            failures.append(entry["path"]); continue
        X.append(vec); y.append(entry["label"])
        if (i + 1) % 200 == 0:
            rate = (i + 1) / (time.time() - t0)
            eta = (len(entries) - (i + 1)) / rate / 60
            print(f"[{split_name}] {i+1}/{len(entries)} — {rate:.2f} videos/sec — ETA {eta:.1f} min")
    X = np.stack(X) if X else np.zeros((0, 768), dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    np.savez(cache_path, X=X, y=y)
    print(f"[{split_name}] done — {len(y)} embedded, {len(failures)} failed")
    return X, y

X_train, y_train = extract_split_features("train", manifest["train"])
X_val, y_val = extract_split_features("val", manifest["val"])
X_test, y_test = extract_split_features("test", manifest["test"])
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-6
norm = lambda X: (X - mu) / sigma

Xtr = torch.tensor(norm(X_train), dtype=torch.float32).to(DEVICE)
ytr = torch.tensor(y_train, dtype=torch.long).to(DEVICE)
Xva = torch.tensor(norm(X_val), dtype=torch.float32).to(DEVICE)
yva = torch.tensor(y_val, dtype=torch.long).to(DEVICE)
Xte = torch.tensor(norm(X_test), dtype=torch.float32).to(DEVICE)
yte = torch.tensor(y_test, dtype=torch.long).to(DEVICE)

class Head(nn.Module):
    def __init__(self, in_dim=768, hidden=256, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden, num_classes))
    def forward(self, x): return self.net(x)

head = Head().to(DEVICE)
optimizer = torch.optim.Adam(head.parameters(), lr=3e-4, weight_decay=1e-4)
class_counts = np.bincount(y_train, minlength=2).astype(np.float32)
class_weights = torch.tensor(class_counts.sum() / (2 * class_counts), dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

real_idx = (ytr == 1).nonzero(as_tuple=True)[0]
fake_idx = (ytr == 0).nonzero(as_tuple=True)[0]

def sample_balanced_batch(bs=128):
    half = bs // 2
    r = real_idx[torch.randint(0, len(real_idx), (half,), device=DEVICE)]
    f = fake_idx[torch.randint(0, len(fake_idx), (bs - half,), device=DEVICE)]
    idx = torch.cat([r, f])
    return Xtr[idx], ytr[idx]

def evaluate(X, y):
    head.eval()
    with torch.no_grad():
        preds = head(X).argmax(dim=1)
        acc = (preds == y).float().mean().item()
        rm, fm = (y == 1), (y == 0)
        rr = (preds[rm] == 1).float().mean().item() if rm.any() else float("nan")
        fr = (preds[fm] == 0).float().mean().item() if fm.any() else float("nan")
    head.train()
    return acc, rr, fr, (rr + fr) / 2

best_bal, best_state = -1, None
for epoch in range(1, 201):
    for _ in range(40):
        xb, yb = sample_balanced_batch()
        optimizer.zero_grad()
        loss = criterion(head(xb), yb)
        loss.backward(); optimizer.step()
    acc, rr, fr, bal = evaluate(Xva, yva)
    if bal > best_bal:
        best_bal, best_state = bal, {k: v.clone() for k, v in head.state_dict().items()}
    if epoch % 20 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} — val_acc {acc:.4f} — val_balanced_acc {bal:.4f} (real_recall {rr:.4f}, fake_recall {fr:.4f})")

print("Best val BALANCED acc:", best_bal)
head.load_state_dict(best_state)
test_acc, test_rr, test_fr, test_bal = evaluate(Xte, yte)
print(f"\nFACE-CROPPED TEST — acc {test_acc:.4f} — balanced_acc {test_bal:.4f} — real_recall {test_rr:.4f} — fake_recall {test_fr:.4f}")
print(f"(Compare to whole-frame baseline: acc 0.5156 — balanced_acc 0.5600 — real_recall 0.6267 — fake_recall 0.4933)")

torch.save(head.state_dict(), "/kaggle/working/video_head_facecrop.pth")
np.savez("/kaggle/working/feature_norm_stats_facecrop.npz", mu=mu, sigma=sigma)
print("Saved video_head_facecrop.pth")

Device: cuda
Manifest missing (session reset) — rebuilding identical split (seed=42)...
Rebuilt manifest saved.
{'train': 4200, 'val': 900, 'test': 900}


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias         

[train] 200/4200 — 0.52 videos/sec — ETA 127.1 min
[train] 400/4200 — 0.48 videos/sec — ETA 133.1 min
[train] 600/4200 — 0.45 videos/sec — ETA 132.1 min
[train] 800/4200 — 0.45 videos/sec — ETA 124.9 min
[train] 1000/4200 — 0.45 videos/sec — ETA 119.4 min
[train] 1200/4200 — 0.45 videos/sec — ETA 111.4 min
[train] 1400/4200 — 0.45 videos/sec — ETA 103.6 min
[train] 1600/4200 — 0.45 videos/sec — ETA 96.6 min
[train] 1800/4200 — 0.45 videos/sec — ETA 88.7 min
[train] 2000/4200 — 0.45 videos/sec — ETA 80.9 min
[train] 2200/4200 — 0.45 videos/sec — ETA 73.6 min
[train] 2400/4200 — 0.45 videos/sec — ETA 66.2 min
[train] 2600/4200 — 0.46 videos/sec — ETA 58.5 min
[train] 2800/4200 — 0.46 videos/sec — ETA 50.9 min
[train] 3000/4200 — 0.46 videos/sec — ETA 43.5 min
[train] 3200/4200 — 0.46 videos/sec — ETA 36.0 min
[train] 3400/4200 — 0.46 videos/sec — ETA 28.7 min
[train] 3600/4200 — 0.46 videos/sec — ETA 21.6 min
[train] 3800/4200 — 0.46 videos/sec — ETA 14.5 min
[train] 4000/4200 — 0.46 vid